In [ ]:
# Тестирование мультизадачной модели на каждом датасете.
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import time
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
from peft import PeftModel


In [ ]:
# Настройка
best_cp = "/content/drive/MyDrive/finetuned_qwen_multitask/checkpoint-9502"

In [ ]:
# Конфигурации датасетов
DATASET_CONFIGS = {

    "Bank": {
        "source": "openml", "openml_id": 1461,
        "task_type": "binary",
        "feature_mappings": {
            'V1':'Age','V2':'Job','V3':'Martial','V4':'Education',
            'V5':'Default','V6':'Balance','V7':'Housing','V8':'Loan',
            'V9':'Contact','V10':'Day of Week','V11':'Month','V12':'Duration',
            'V13':'Campaign','V14':'Pdays','V15':'Previous','V16':'Poutcome',
            'Class':'y'
        },
        "prompt_config": {
            "task": "Predict whether a bank client will subscribe",
            "labels": ["no", "yes"],
            "entity": "Client",
            "question": "Will this client subscribe to a term deposit?"
        },
    },

    "Blood": {
        "source": "openml", "openml_id": 1464,
        "task_type": "binary",
        "feature_mappings": {
            'V1':'Recency','V2':'Frequency','V3':'Monetary','V4':'Time',
            'Class':'Donated blood'
        },
        "prompt_config": {
            "task": "Predict whether a person donated blood",
            "labels": ["no", "yes"],
            "entity": "Donor",
            "question": "Did this person donate blood?"
        },
    },

    "California": {
        "source": "openml", "openml_id": 44090,
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether house price is above median",
            "labels": ["no", "yes"],
            "entity": "House",
            "question": "Is this house price above median?"
        },
    },

    "Car": {
        "source": "openml", "openml_id": 40975,
        "task_type": "multiclass",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict car evaluation",
            "labels": ["unacceptable", "acceptable", "good", "very good"],
            "entity": "Car",
            "question": "What is the evaluation of this car?"
        },
    },

    "Credit_G": {
        "source": "openml", "openml_id": 31,
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Classify credit risk as good or bad",
            "labels": ["bad", "good"],
            "entity": "Client",
            "question": "Is this client a good credit risk?"
        },
    },

    "Diabetes": {
        "source": "kaggle",
        "kaggle_dataset": "uciml/pima-indians-diabetes-database",
        "kaggle_file": "diabetes.csv",
        "target_col": "Outcome",
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether a patient has diabetes",
            "labels": ["no", "yes"],
            "entity": "Patient",
            "question": "Does this patient have diabetes?"
        },
    },

    "Heart": {
        "source": "kaggle",
        "kaggle_dataset": "fedesoriano/heart-failure-prediction",
        "kaggle_file": "heart.csv",
        "target_col": "HeartDisease",
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether a patient has heart disease",
            "labels": ["no", "yes"],
            "entity": "Patient",
            "question": "Does this patient have heart disease?"
        },
    },

    "Income": {
        "source": "openml", "openml_id": 1590,
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether a person earns more than 50K a year",
            "labels": ["<=50K", ">50K"],
            "entity": "Person",
            "question": "Does this person earn more than 50K a year?"
        },
    },

    "Jungle": {
        "source": "openml", "openml_id": 41027,
        "task_type": "multiclass",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict the endgame result of Jungle Chess",
            "labels": ["white_win", "draw", "black_win"],
            "entity": "Game Position",
            "question": "What is the game result?"
        },
    },
}

In [ ]:
# Загрузка данных
def load_openml(cfg):
    dataset = fetch_openml(data_id=cfg["openml_id"], as_frame=True, parser='auto')
    df = dataset.data.copy()
    y  = dataset.target

    if cfg["feature_mappings"]:
        df = df.rename(columns=cfg["feature_mappings"])
        feature_names = list(cfg["feature_mappings"].values())[:-1]
        target_name   = list(cfg["feature_mappings"].values())[-1]
    else:
        feature_names = df.columns.tolist()
        target_name   = y.name

    df[target_name] = y
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])
    return df, feature_names, target_name


def load_kaggle(cfg):
    import kagglehub
    path = kagglehub.dataset_download(cfg["kaggle_dataset"])
    df = pd.read_csv(f"{path}/{cfg['kaggle_file']}")
    target_name   = cfg["target_col"]
    feature_names = [c for c in df.columns if c != target_name]
    le = LabelEncoder()
    df[target_name] = le.fit_transform(df[target_name])
    return df, feature_names, target_name


def get_test_split(df, target_name, seed=42):
    # Воспроизводим тот же сплит, что и при обучении
    tv, test = train_test_split(df, test_size=0.2, random_state=seed, stratify=df[target_name])
    return test


def load_all_test_sets():
    """Загрузка тестовых выборок всех датасетов"""
    test_splits = {}

    for name, cfg in DATASET_CONFIGS.items():
        print(f"Dataset: {name}")

        if cfg["source"] == "openml":
            df, feature_names, target_name = load_openml(cfg)
        else:
            df, feature_names, target_name = load_kaggle(cfg)

        test_df = get_test_split(df, target_name)

        test_splits[name] = (test_df, feature_names, target_name)
        print(f"Test={len(test_df)}")

    return test_splits

# 2. Вспомогательные функции

In [ ]:
# Промпт и инференс
def row_to_text_template(row, feature_names):
    template_parts = []
    for feature in feature_names:
        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    return " ".join(template_parts)

In [ ]:
def parse_prediction(response, labels):
    response = response.lower().strip().strip("'\"")
    response = response.rstrip('.,!? ').strip("'\"").strip()

    for i, class_name in enumerate(labels):
        class_lower = class_name.lower().strip().strip("'\"")

        if response == class_lower:
            return i

        if response.startswith(class_lower):
            return i

        if class_lower in response.split():
            return i

    print(f"Warning: Could not parse '{response}' (expected one of {labels})")
    return 0

In [ ]:
def compute_metrics(y_true, y_pred, y_prob, task_type):
    avg = "binary" if task_type == "binary" else "macro"

    metrics = {
        "F1": f1_score(y_true, y_pred, average=avg, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average=avg, zero_division=0),
        "Recall": recall_score(y_true, y_pred, average=avg, zero_division=0),
    }

    try:
        mc  = "raise" if task_type == "binary" else "ovr"
        ypr = np.array(y_prob)
        metrics["ROC-AUC"] = roc_auc_score(
            y_true, ypr, multi_class=mc, average="macro")
    except:
        metrics["ROC-AUC"] = 0.0

    return metrics

In [ ]:
def bootstrap_metrics(y_true, y_pred, y_prob, task_type, n_iter=1000):
    scores = []
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    for i in range(n_iter):
        yt, yp, ypr = resample(y_true, y_pred, y_prob, random_state=i+1)
        try:
            m = compute_metrics(yt, yp, ypr, task_type)
            scores.append([m["ROC-AUC"], m["F1"], m["Accuracy"], m["Precision"], m["Recall"]])
        except Exception:
            continue

    if not scores:
        return {k: "0.0000±0.0000" for k in ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]}

    scores = np.array(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    keys = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]
    return {k: f"{m:.4f}±{s:.4f}" for k, m, s in zip(keys, means, stds)}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#Загрузка модели Qwen 3.0 4B Instruct
model_name = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизаторов
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Загрузка модель с 16-битной квантизацией и автоматическим распределением по устройствам
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="sdpa"
)

model = PeftModel.from_pretrained(base_model, best_cp).to(device).eval()

if torch.cuda.is_available():
    print(f"Использовано памяти: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Using device: cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Использовано памяти: 8.18 GB


In [ ]:
def create_prompt(row, feature_names, target_name, dataset_name, prompt_config, tokenizer, include_answer=False):

    labels_str = "', '".join(prompt_config['labels'])

    system_prompt = (
        f"You are a classifier. Dataset: {dataset_name}. Task: {prompt_config['task']}.\n"
        f"Answer with only one word from: '{labels_str}'."
    )

    user_message = (
        f"Dataset: {dataset_name}. {prompt_config['entity']} information: "
        f"{row_to_text_template(row, feature_names)}\n"
        f"{prompt_config['question']}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message},
    ]

    # Если это данные для обучения, добавляем ответ ассистента
    if include_answer:
        target_value = row[target_name]
        answer = prompt_config['labels'][int(target_value)]
        messages.append({"role": "assistant", "content": answer})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=not include_answer
    )

In [ ]:
def predict_single_with_prob(prompt, feature_names, target_name, dataset_name, prompt_config, model, tokenizer, device, max_new_tokens=5):
    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True,
        )

    # Декодирование текста
    generated_ids  = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для всех классов
    first_token_logits = outputs.scores[0][0]

    labels_ids = []
    for labels_name in prompt_config['labels']:
        labels_id = tokenizer.encode(labels_name, add_special_tokens=False)[0]
        labels_ids.append(labels_id)

    labels_logits = torch.stack([first_token_logits[cid] for cid in labels_ids])
    probs = F.softmax(labels_logits, dim=0)

    # Возвращаем ответ и вероятности всех классов
    return response, probs.cpu().numpy()

In [ ]:
def test_dataset(model, test_df, feature_names, target_name, dataset_name, tokenizer, device):
    cfg = DATASET_CONFIGS[dataset_name]
    prompt_config = cfg["prompt_config"]
    task_type = cfg["task_type"]

    print(f"Тестирование: {dataset_name} (размер теста: {len(test_df)})")

    y_true, y_pred, y_prob = [], [], []
    times = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=dataset_name):
        t0 = time.time()
        prompt = create_prompt(row, feature_names, target_name, dataset_name, prompt_config, tokenizer)
        response, probs = predict_single_with_prob(prompt, feature_names, target_name, dataset_name, prompt_config, model, tokenizer, device)
        times.append(time.time() - t0)

        y_true.append(int(row[target_name]))
        y_pred.append(parse_prediction(response, prompt_config['labels']))
        if task_type == "binary":
            y_prob.append(float(probs[1]))
        else:
            y_prob.append(probs)

    # Промежуточный отчёт по времени
    total_time = sum(times)
    print(f"Время инференса : {total_time:.1f}с")
    # Вычисление метрик
    metrics = compute_metrics(y_true, y_pred, y_prob, task_type)
    boot = bootstrap_metrics(y_true, y_pred, y_prob, task_type, n_iter=1000)

    print(f"ROC-AUC: {metrics['ROC-AUC']:.4f}  ({boot['ROC-AUC']})")
    print(f"F1: {metrics['F1']:.4f}  ({boot['F1']})")
    print(f"Accuracy: {metrics['Accuracy']:.4f}  ({boot['Accuracy']})")
    print(f"Precision: {metrics['Precision']:.4f}  ({boot['Precision']})")
    print(f"Recall: {metrics['Recall']:.4f}  ({boot['Recall']})")

    return {
        "metrics": metrics,
        "bootstrap": boot,
        "time_total": total_time,
        "time_per_sample": np.mean(times),
        "task_type": task_type,
        "n_classes": len(prompt_config['labels']),
        "test_size": len(test_df),
    }

In [ ]:
# Загрузка всех тестовых выборок
test_splits = load_all_test_sets()

# Тестирование каждого датасета
all_results = {}
total_start = time.time()

for name, (test_df, feature_names, target_name) in test_splits.items():
    all_results[name] = test_dataset(model, test_df, feature_names, target_name, name, tokenizer, device)

print(f"\nОбщее время тестирования: {(time.time()-total_start)/60:.1f} мин")

Dataset: Bank
Test=9043
Dataset: Blood
Test=150
Dataset: California
Test=4127
Dataset: Car
Test=346
Dataset: Credit_G
Test=200
Dataset: Diabetes
Test=154
Dataset: Heart
Using Colab cache for faster access to the 'heart-failure-prediction' dataset.
Test=184
Dataset: Income
Test=9769
Dataset: Jungle
Test=8964
Тестирование: Bank (размер теста: 9043)


Bank:   0%|          | 0/9043 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Время инференса : 1963.4с
ROC-AUC: 0.9234  (0.9233±0.0038)
F1: 0.6068  (0.6072±0.0117)
Accuracy: 0.8905  (0.8905±0.0033)
Precision: 0.5233  (0.5235±0.0134)
Recall: 0.7221  (0.7229±0.0137)
Тестирование: Blood (размер теста: 150)


Blood:   0%|          | 0/150 [00:00<?, ?it/s]

...

Время инференса : 40.0с
ROC-AUC: 0.9099  (0.9082±0.0238)
F1: 0.8438  (0.8418±0.0284)
Accuracy: 0.8370  (0.8357±0.0275)
Precision: 0.9000  (0.8985±0.0316)
Recall: 0.7941  (0.7931±0.0401)
Тестирование: Income (размер теста: 9769)


Income:   0%|          | 0/9769 [00:00<?, ?it/s]

Время инференса : 5201.0с
ROC-AUC: 0.9276  (0.9276±0.0027)
F1: 0.7209  (0.7207±0.0069)
Accuracy: 0.8482  (0.8482±0.0036)
Precision: 0.6437  (0.6434±0.0086)
Recall: 0.8191  (0.8191±0.0078)
Тестирование: Jungle (размер теста: 8964)


Jungle:   0%|          | 0/8964 [00:00<?, ?it/s]

Время инференса : 2753.1с
ROC-AUC: 0.9764  (0.9764±0.0009)
F1: 0.8268  (0.8269±0.0047)
Accuracy: 0.8607  (0.8608±0.0036)
Precision: 0.8001  (0.8002±0.0048)
Recall: 0.8901  (0.8902±0.0033)


Общее время тестирования: 186.1 мин


In [ ]:
all_results

{'Bank': {'metrics': {'F1': 0.6068308181096108,
   'Accuracy': 0.8905230565077961,
   'Precision': 0.5232876712328767,
   'Recall': 0.722117202268431,
   'ROC-AUC': np.float64(0.9233686034660926)},
  'bootstrap': {'ROC-AUC': '0.9233±0.0038',
   'F1': '0.6072±0.0117',
   'Accuracy': '0.8905±0.0033',
   'Precision': '0.5235±0.0134',
   'Recall': '0.7229±0.0137'},
  'time_total': 1963.4060759544373,
  'time_per_sample': np.float64(0.21711888487829673),
  'task_type': 'binary',
  'n_classes': 2,
  'test_size': 9043},
 'Blood': {'metrics': {'F1': 0.48,
   'Accuracy': 0.74,
   'Precision': 0.46153846153846156,
   'Recall': 0.5,
   'ROC-AUC': np.float64(0.7348927875243664)},
  'bootstrap': {'ROC-AUC': '0.7384±0.0464',
   'F1': '0.4786±0.0716',
   'Accuracy': '0.7422±0.0340',
   'Precision': '0.4623±0.0803',
   'Recall': '0.5030±0.0832'},
  'time_total': 32.66870188713074,
  'time_per_sample': np.float64(0.2177913459142049),
  'task_type': 'binary',
  'n_classes': 2,
  'test_size': 150},
 'Cal